# Assessing_stigma.ipynb

**Author:** Zane Zhang

**Date:** 2025/07/09 18:29

**Description:** 

The notebook is the last part of the whole research, in which we will assess stigma in response to Depression Misinformation. We adapt a strategy used by Li et al. (2018) while we use python programs to complete this task instead of the WEKA software.

Before we start to build a classifier to differentiate depression stigma and non-stigma, we first label 100 comments manually as the training subset.

In [ ]:
# Sample 100 comments randomly from DMisinfo
import pandas as pd

df = pd.read_csv("data/annotation_sample.csv")
post_ids = set(df["Post_id"])
comments = pd.read_csv("data/comments_with_uid.csv")
dmisinfo_comment = comments[comments["Post_id"].isin(post_ids)]

sample = dmisinfo_comment.sample(n=200, random_state=42).reset_index(drop=True)
sample["stigma"] = ""   # stigma=1, non-stigma=0
sample[["uid", "Post_id", "Sentence_normalized", "stigma"]].to_csv("data/stigma_sample.csv", index=False, encoding="utf-8")

Then we label these comments manually. The coding scheme (Griffith, 2004) is showed below:

<img src="data/stigma_frame.png" style="width:50%">

After preparation of training data, we start the stigma detection. In this stage, we compare two methods to do our research. First, we adapt the MetaLLaMa-chat-13B model to label the remaining comments, and evaluate it by F1-score, precision, and recall.

What we should notice is that the evaluation method of MentaLLaMa is **based on a small number of manually labeled comparisons**, which is different from RoBERTa. Therefore, we need to calculate metrics within the manually labeled sample.

In [ ]:
# 5-shots (3 stigma, 2 non-stigma)
from google.colab import drive
drive.mount("/content/drive")
import pandas as pd

sample = pd.read_excel("drive/MyDrive/DMisinfo/stigma_annotation.xlsx")
# sample_1 = sample[sample["stigma"] == 1].sample(n=3, random_state=42)
# 人工选择3条合适的stigma
sample_1 = pd.read_excel("drive/MyDrive/DMisinfo/stigma_3_shot.xlsx")
sample_0 = sample[sample["stigma"] == 0].sample(n=2, random_state=42)
sample = pd.concat([sample_1, sample_0]).sample(frac=1, random_state=42)    # disorder

texts = sample["Sentence_normalized"].tolist()
sample["answer"] = sample["stigma"].apply(lambda x: "Yes" if x == 1 else "No")
answers = sample["answer"].tolist()

examples = ""
for i, (text, answer) in enumerate(zip(texts, answers), 1):
    examples += f"Example {i}: Comment: {text} Answer: {answer}\n"

# Save examples
with open("drive/MyDrive/DMisinfo/prompt_examples.txt", "w", encoding="utf-8") as f:
    f.write(examples)

In [ ]:
# Load model
from transformers import LlamaTokenizer, LlamaForCausalLM
import torch

model_path = "klyang/MentaLLaMa-chat-13B"

tokenizer = LlamaTokenizer.from_pretrained(model_path)
model = LlamaForCausalLM.from_pretrained(
    model_path,
    device_map="auto",  # auto select CPU/GPU
    torch_dtype=torch.float16   # half precision to save memory
)
model.eval()

In [ ]:
# Classifier
from google.colab import drive
drive.mount("/content/drive")

examples = open("drive/MyDrive/DMisinfo/prompt_examples.txt", "r", encoding="utf-8").read()
frame = """1 People with depression could snap out of it they wanted
2 Depression is a sign of personal weakness
3 Depression is not a real medical illness
4 People with depression are dangerous
5 It is best to avoid people with depression so you don't become depressed yourself
6 People with depression are unpredictable
7 If I had depression I would not tell anyone
8 I would not employ someones if I knew they had been depressed
9 I would not vote for a politician if I knew they had been depressed
10 People with depression are glorified
11 People with depression are self-centred"""

# prompt = f"""You are an expert in psychiatry and social media data. Please follow the frame where we determine whether a Reddit comment actively generates stigma towards people with depression:
# {frame}
# Now, given the frame above, please determine whether the comment actively generates stigma against people with depression. Start your answer with yes or no, and then give your reasoning. Answer yes if any term in the frame is satisfied. Answer no if the comment’s author only talks about their experience:
# Comment: {comment} Answer:
# """
# prompt = f"""You are an expert in psychiatry and social media data. Please analyze the following examples where we determine whether a Reddit comment actively generates stigma towards people with depression. A comment is not stigmatizing if it only talks about one’s personal experience with mental health. There are five examples, one on each line. Each example contains the comment and the human-generated answer:
# {examples}
# Now, given what you learned from the five examples above, please determine whether the comment actively generates stigma against people with depression. Start your answer with yes or no, and then give your reasoning. Answer no if the comment’s author only talks about their experience:
# Comment: {comment} Answer:
# """
prompt_prefix = f"""You are an expert in psychiatry and social media data. Please follow the frame where we determine whether a Reddit comment actively generates stigma towards people with depression:
    {frame}
    And here are five examples:
    {examples}
    Now, given the frame and what you learned from examples above, please determine whether the comment actively generates stigma against people with depression. Start your answer with yes or no, and then give your reasoning. Answer yes if any term in the frame is satisfied. Answer no if the comment’s author only talks about their experience:
    """

def classify(comment, max_tokens=128):
    prompt = f"{prompt_prefix}\nComment: {comment} Answer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generate_ids = model.generate(
            inputs.input_ids,
            max_new_tokens=max_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            temperature=1,
            top_p=None
        )

    response = tokenizer.decode(generate_ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return response.strip()

In [ ]:
# Labeling manually labeled datasets using LLM
import pandas as pd
from tqdm.auto import tqdm
import re

annotation = pd.read_excel("drive/MyDrive/DMisinfo/stigma_annotation.xlsx")

def extract_label(output):
    if "yes" in output.lower():
      return 1
    elif "the comment actively generates stigma" in output.lower():
      return 1
    else:
      return 0

results = []

for _, row in tqdm(annotation.iterrows(), total=len(annotation)):
      output = classify(row.Sentence_normalized)
      label = extract_label(output)
      results.append({
          "uid": row.uid,
          "Sentence_normalized": row.Sentence_normalized,
          "stigma": row.stigma,
          "output": output,
          "label": label
      })

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv(
    "drive/MyDrive/DMisinfo/stigma_machine_comparison.csv",
    index=False, encoding="utf-8"
)

In [ ]:
# Evaluation
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

comparison = pd.read_csv("drive/MyDrive/DMisinfo/stigma_machine_comparison.csv")
# 剔除shot
# uids = pd.read_csv("drive/MyDrive/DMisinfo/stigma_examples_uids.csv")
# uids = set(uids["uid"])
# comparison = comparison[~comparison["uid"].isin(uids)]

def evaluate_model(y_true, y_pred, model_name="Model"):
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_pos = f1_score(y_true, y_pred, pos_label=1)
    prec = precision_score(y_true, y_pred, pos_label=1)
    rec = recall_score(y_true, y_pred, pos_label=1)

    print(f"{model_name:<12} | {f1_macro:10.4f} | {f1_pos:7.4f} | {prec:6.4f} | {rec:6.4f}")

evaluate_model(comparison["stigma"], comparison["label"], model_name="MentaLLaMa-chat-13B")

Then we prepare the unlabeled data for LLM classification.

In [ ]:
# Prepared comments
import math
import pandas as pd

comments = pd.read_csv("drive/MyDrive/DMisinfo/comments_with_labels.csv")

batch_size = 1000
num_batches = math.ceil(len(comments) / batch_size)

for i in range(num_batches):
    start = i * batch_size
    end = min((i + 1) * batch_size, len(comments))
    batch = comments.iloc[start:end]
    batch.to_csv(f"drive/MyDrive/DMisinfo/batches/comments_batch_{i+1:03d}.csv", index=False, encoding="utf-8")

In [ ]:
# Classify comments
import os
from tqdm.auto import tqdm

# Access to completed document numbers
processed = set()
result_dir = "drive/MyDrive/DMisinfo/mentallama"

for fname in os.listdir(result_dir):
    if fname.startswith("comments_batch_") and fname.endswith(".csv"):
        try:
            batch_num = int(fname.replace("comments_batch_", "").replace(".csv", ""))
            processed.add(batch_num)
        except:
            continue

# Proceed with the remaining batches
def extract_label(output):
    if "yes" in output.lower():
      return 1
    elif "the comment actively generates stigma" in output.lower():
      return 1
    else:
      return 0

for i in range(num_batches):
    batch_id = i + 1  # Document numbering starts at 001
    if batch_id in processed:
        print(f"Batch {batch_id:03d} already processed. Skipping.")
        continue

    print(f"Processing Batch {batch_id:03d}")
    batch = pd.read_csv(f"drive/MyDrive/DMisinfo/batches/comments_batch_{batch_id:03d}.csv")
    results = []
    total = len(batch)

    for _, row in tqdm(enumerate(batch.itertuples(), 1), total=total):
        output = classify(row.Sentence_normalized)
        label = extract_label(output)
        results.append({
          "uid": row.uid,
          "Sentence_normalized": row.Sentence_normalized,
          "output": output,
          "label": label
        })

    # Save results
    pd.DataFrame(results).to_csv(
        f"drive/MyDrive/DMisinfo/mentallama/comments_batch_{batch_id:03d}.csv",
        index=False, encoding="utf-8"
    )

In [ ]:
# Combine all results
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import os

result_dir = "drive/MyDrive/DMisinfo/mentallama"
all_results = []

# Read every batch
for fname in os.listdir(result_dir):
    if fname.startswith("comments_batch_") and fname.endswith(".csv"):
        df = pd.read_csv(os.path.join(result_dir, fname))
        all_results.append(df)

comments_labeled_by_mentallama = pd.concat(all_results, ignore_index=True)

comments_labeled_by_mentallama.to_csv("drive/MyDrive/DMisinfo/comments_labeled_by_mentallama.csv")

After labeled and combine all data, we start to analyze the difference between responses to misinformation and non-misinformation, using $\chi^2$ test

In [2]:
import pandas as pd
from scipy.stats import chi2_contingency

df = pd.read_csv("data/comments_labeled_by_mentallama.csv")
post = pd.read_csv("data/comments_with_labels.csv")
post = post.rename(columns={"label_pred": "post_label"})

# Merge post label and comment label using uid
df = df.merge(post, on="uid", how="inner")

# Chi-squared test
table = pd.crosstab(df["post_label"], df["label"])
chi2, p, dof, expected = chi2_contingency(table)

# calculate percentage
# stigma in comments to misinformation post
pct_misinfo = (df[(df["post_label"] == 1) & (df["label"] == 1)].shape[0] / df[df["post_label"] == 1].shape[0]) * 100
# stigma in comments to non-misinformation post
pct_non_misinfo = (df[(df["post_label"] == 0) & (df["label"] == 1)].shape[0] / df[df["post_label"] == 0].shape[0]) * 100

# significance
if p < 0.001:
    significance = "***"
elif p < 0.01:
    significance = "**"
elif p < 0.05:
    significance = "*"
else:
    significance = ""

# Output
print(f"{'Metric':<10} {'% Misinfo':<12} {'% NMisinfo':<12} {'p-value':<10} {'χ²'}")
print(f"{'Stigma':<10} {pct_misinfo:>5.1f}%{'':<6} {pct_non_misinfo:>5.1f}%{'':<5} {significance:<10} {chi2:.1f}")

Metric     % Misinfo    % NMisinfo   p-value    χ²
Stigma      11.0%         9.3%      ***        12.9
